# 使用 Microsoft Foundry 微调模型

本笔记本遵循当前的 [Microsoft Foundry 微调工作流程](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning?WT.mc_id=academic-105485-koreyst)。它使用指向您 Foundry 资源的 `/openai/v1/` 端点的 **OpenAI Python SDK (v1)**，因此相同的代码只需更改客户端设置即可在 OpenAI 平台上运行。

> <strong>先决条件</strong>
> - 一个 [Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) 项目，拥有 **Foundry 所有者** 角色（微调和部署所需）。
> - 运行 `pip install "openai>=1.0" python-dotenv`
> - 一个包含 `AZURE_OPENAI_ENDPOINT` 和 `AZURE_OPENAI_API_KEY` 的 `.env` 文件（参见[课程设置指南](./../../../00-course-setup/02-setup-local.md?WT.mc_id=academic-105485-koreyst)）。
> - 一个当前支持的基础模型，例如 `gpt-4.1-mini`（参见[可微调模型列表](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst#fine-tuning-models)）。

微调通过在与你的任务相关的额外示例上重新训练基础模型来提升其性能。提示工程技术如 _少样本学习_ 和 _检索增强生成_ 可以用相关数据增强提示，但受限于模型的上下文窗口大小。

通过微调，您可以重新训练模型本身（使用比上下文窗口容纳更多的示例），并部署一个 _自定义_ 版本，该版本在推理时不再需要那些示例。这提升了质量，释放了上下文窗口，并且通过缩短提示可以降低成本和延迟。在底层，Foundry 使用 **LoRA（低秩适配）** 进行高效训练。

Foundry 支持三种技术：本教程使用的 **监督微调 (SFT)**，以及 **DPO**（偏好对齐）和 **RFT**（强化微调）。工作流程包括四个步骤：

1. 准备并上传您的训练和验证数据。
2. 运行训练作业以创建微调模型。
3. 监控作业，查看指标并选择检查点。
4. 部署微调模型并用于推理。

在本教程中，我们使用 SFT 微调 `gpt-4.1-mini`，构建一个以打油诗形式回答元素周期表问题的聊天机器人。

---


### 第1.1步：准备你的数据集

让我们构建一个聊天机器人，通过用打油诗回答关于元素的问题，帮助你理解元素周期表。在_本_简单教程中，我们将仅创建一个数据集，用一些示例响应展示数据的预期格式来训练模型。在真实使用场景中，你需要创建一个包含更多示例的数据集。你也可以使用现有的开源数据集（针对你的应用领域），并对其重新格式化以用于微调。

由于我们聚焦于`gpt-4.1-mini`并且期望单轮回复（聊天补全），我们可以使用[该建议格式](https://platform.openai.com/docs/guides/fine-tuning/preparing-your-dataset?WT.mc_id=academic-105485-koreyst)创建示例，它符合OpenAI聊天补全的要求。如果你预期多轮对话内容，则使用[多轮示例格式](https://platform.openai.com/docs/guides/fine-tuning/multi-turn-chat-examples?WT.mc_id=academic-105485-koreyst)，其中包含一个`weight`参数，用来指示哪些消息应当（或不应当）用于微调过程。

本教程中我们将使用更简单的单轮格式。数据采用[jsonl格式](https://jsonlines.org/?WT.mc_id=academic-105485-koreyst)，每行一条记录，每条记录是一个JSON格式的对象。下面的片段展示了两个示例记录——完整示例集（10个示例）见[training-data.jsonl](../../../../../18-fine-tuning/python/openai/training-data.jsonl)，这是我们用于微调教程的样本。**注意：** 每条记录_必须_定义在单行内（不能像格式化的JSON文件那样分多行）。

```json
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Gallium"}, {"role": "assistant", "content": "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}]}
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Hydrogen"}, {"role": "assistant", "content": "Hydrogen, the first in the line - The lightest of all, so divine - It's in water, you see - And in stars, it's the key - The universe's most common sign"}]}
```

在真实使用中，你需要更多示例以获得较好效果——权衡点在于回复质量与微调的时间/成本。我们这里用的是小型示例集，以便快速完成微调并演示流程。更多复杂微调教程见[OpenAI示例集](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst)。


---

### 第 1.2 步：上传您的数据集

使用 Files API（`purpose="fine-tune"`）将您的训练和验证文件上传到 Foundry。提供<strong>验证文件</strong>可以让 Foundry 报告验证损失，从而帮助您发现过拟合现象。

在运行以下代码之前，请确保您已经：
 - 安装了 SDK：`pip install "openai>=1.0" python-dotenv`
 - 创建了包含 `AZURE_OPENAI_ENDPOINT` 和 `AZURE_OPENAI_API_KEY` 的 `.env` 文件

该代码会创建一个指向您 Foundry 资源的 `/openai/v1/` 端点的 OpenAI 客户端，然后上传这两个文件并打印它们的 ID。


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Point the OpenAI SDK at your Microsoft Foundry resource's /openai/v1/ endpoint.
# (For the OpenAI platform instead, use: client = OpenAI()  with OPENAI_API_KEY set.)
client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
)

training_response = client.files.create(
    file=open("./training-data.jsonl", "rb"), purpose="fine-tune"
)
validation_response = client.files.create(
    file=open("./validation-data.jsonl", "rb"), purpose="fine-tune"
)

training_file_id = training_response.id
validation_file_id = validation_response.id

print("Training file ID:", training_file_id)
print("Validation file ID:", validation_file_id)


---

### 第2.1步：使用SDK创建微调作业


In [ ]:
# Create the fine-tuning job.
# trainingType "GlobalStandard" is the recommended tier (lower cost, faster queue).
# Use "Standard" if you need regional data residency, or "Developer" for cheap experiments.
job = client.fine_tuning.jobs.create(
    model="gpt-4.1-mini",              # base model to fine-tune
    training_file=training_file_id,
    validation_file=validation_file_id,
    suffix="elements-limerick",        # helps you identify the resulting model
    seed=105,                          # makes the run reproducible
    extra_body={"trainingType": "GlobalStandard"},
)

job_id = job.id
print("Fine-tuning Job ID:", job_id)
print("Status:", job.status)


---

### 第2.2步：检查作业状态

下面是你可以用 `client.fine_tuning.jobs` API 做的一些事情：
- `client.fine_tuning.jobs.list(limit=<n>)` - 列出最近的n个微调作业
- `client.fine_tuning.jobs.retrieve(<job_id>)` - 获取特定作业的详细信息
- `client.fine_tuning.jobs.cancel(<job_id>)` - 取消一个作业
- `client.fine_tuning.jobs.list_events(fine_tuning_job_id=<job_id>, limit=<n>)` - 列出该作业的事件
- `client.fine_tuning.jobs.checkpoints.list(<job_id>)` - 列出可部署的检查点（最近几个epoch）

当作业开始时，Foundry首先会_验证训练文件_以确保数据格式正确。然后进行训练，时长可能从几分钟到几小时不等，取决于模型和数据集的大小。


In [ ]:
# List the last 10 fine-tuning jobs
client.fine_tuning.jobs.list(limit=10)

# Retrieve the current state of your fine-tuning job
client.fine_tuning.jobs.retrieve(job_id)

# List up to 10 events from the job
client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10)


In [ ]:
# Track the job status until it is complete
response = client.fine_tuning.jobs.retrieve(job_id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)


---

### 第2.3步：跟踪事件以监控进度


In [ ]:
# Track progress in a more granular way by checking events.
# Re-run this cell until you see "The job has successfully completed".
response = client.fine_tuning.jobs.list_events(job_id)

events = response.data
events.reverse()

for event in events:
    print(event.message)


In [ ]:
# When the job finishes, the last few epochs are available as deployable checkpoints.
# Best practice: don't blindly deploy the final epoch - review the checkpoints and pick
# the one that generalizes best (watch train_loss vs. valid_loss and token accuracy).
checkpoints = client.fine_tuning.jobs.checkpoints.list(job_id)
for cp in checkpoints.data:
    print(cp.fine_tuned_model_checkpoint, "| step:", cp.step_number)


### 第2.4步：在Foundry门户中查看状态、指标和检查点


您还可以在 [Microsoft Foundry 门户](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) 的 **构建 > 微调** 下跟踪作业。选择您的作业以查看其状态、训练事件、超参数和指标。请关注以下指标：

- `train_loss` 和 `full_valid_loss` - 应随着时间减少。
- `train_mean_token_accuracy` 和 `full_valid_mean_token_accuracy` - 应该增加。

如果训练曲线和验证曲线出现分歧，您可能出现了过拟合——尝试减少训练周期数或降低学习率乘数。下面的示意图展示了您将看到的状态、消息和指标面板（具体界面因提供商而异）。

![Fine-tuning job status](../../../../../translated_images/zh-CN/fine-tuned-model-status.563271727bf7bfba.webp)


您还可以通过向下滚动视觉仪表板来查看状态消息和指标，如下所示：

| Messages | Metrics |
|:---|:---|
| ![Messages](../../../../../translated_images/zh-CN/fine-tuned-messages-panel.4ed0c2da5ea1313b.webp) |  ![Metrics](../../../../../translated_images/zh-CN/fine-tuned-metrics-panel.700d7e4995a65229.webp)|


---

### 步骤 3.1：检索微调模型 ID

当作业成功时，`response.fine_tuned_model` 保存您的自定义模型 ID（例如，`gpt-4.1-mini-2025-04-14.ft-...`）。在 Azure 上，您需要先<strong>部署</strong>该模型，然后才能调用它。


In [ ]:
# Retrieve the id of the fine-tuned model once the job has succeeded
response = client.fine_tuning.jobs.retrieve(job_id)
fine_tuned_model_id = response.fine_tuned_model
print("Fine-tuned Model ID:", fine_tuned_model_id)


### 第3.2步：部署微调模型

与OpenAI平台（您可以直接调用微调模型ID）不同，Microsoft Foundry要求您先<strong>部署</strong>模型。最简单的方法是使用[Foundry门户](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)：进入<strong>构建 > 微调</strong>，选择已完成的作业，然后选择<strong>部署</strong>。为部署命名（例如，`elements-limerick`）——代码中调用的就是这个部署名称。

您也可以使用控制平面REST/ARM API通过编程方式部署；请参阅[部署指南](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning-deploy?WT.mc_id=academic-105485-koreyst)。请记住，已部署的自定义模型按小时计费，且非活动部署将在15天后被移除。


In [ ]:
# Once deployed, call your fine-tuned model by its DEPLOYMENT name via the Responses API.
# (On the OpenAI platform you'd pass fine_tuned_model_id directly instead.)
deployment_name = "elements-limerick"  # the name you gave the deployment in Foundry

completion = client.responses.create(
    model=deployment_name,
    input=[
        {"role": "system", "content": "You are Elle, a factual chatbot that answers questions about elements in the periodic table with a limerick"},
        {"role": "user", "content": "Tell me about Strontium"},
    ],
    store=False,
)
print(completion.output_text)


---

### 步骤 3.3：在 Foundry 游乐场测试你的微调模型

你也可以在 [Microsoft Foundry 门户](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) <strong>游乐场</strong> 中测试已部署的模型 —— 从模型下拉菜单中选择你的微调部署，并尝试一个提示。使用你训练时的<strong>相同系统消息</strong>；不同的系统消息会改变模型的行为。

> **提示：** 将微调模型与基础的 `gpt-4.1-mini` 并排比较。注意微调模型如何从你的示例中返回押韵诗格式的答案，而基础模型只是遵循系统提示。

**这是一个故意简单的示例来展示流程，而非真实世界的数据集。** 在生产环境中，你会用真实数据进行微调（例如，用于客户服务的产品目录），那时质量差异会更加明显 —— 而仅通过提示工程来达到那种质量会消耗更多的调用令牌。欲了解更完整的示例，请参阅 [OpenAI Cookbook 微调指南](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst) 和 [Foundry 微调教程](https://learn.microsoft.com/azure/ai-foundry/openai/tutorials/fine-tune?WT.mc_id=academic-105485-koreyst)。

---


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
